In [1]:
!pip install transformers datasets torch evaluate seqeval Pillow matplotlib -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

df = load_dataset('nielsr/funsd-layoutlmv3')


print(f'Train samples : {len(df["train"])}')
print(f'Test samples  : {len(df["test"])}')

sample = df['train'][0]
print(f'\nFirst sample words : {sample["tokens"][:5]}')
print(f'First sample labels: {sample["ner_tags"][:5]}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/770 [00:00<?, ?B/s]

funsd/train-00000-of-00001.parquet:   0%|          | 0.00/26.3M [00:00<?, ?B/s]

funsd/test-00000-of-00001.parquet:   0%|          | 0.00/9.54M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/149 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50 [00:00<?, ? examples/s]

Train samples : 149
Test samples  : 50

First sample words : ['R&D', ':', 'Suggestion:', 'Date:', 'Licensee']
First sample labels: [0, 3, 3, 3, 5]


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

ID2LABEL = {0:'O', 1:'B-HEADER', 2:'I-HEADER',
            3:'B-QUESTION', 4:'I-QUESTION',
            5:'B-ANSWER', 6:'I-ANSWER'}

all_labels = []
for s in df['train']:
    all_labels.extend([ID2LABEL[t] for t in s['ner_tags']])

label_counts = Counter(all_labels)
labels = list(label_counts.keys())
counts = list(label_counts.values())
colors = ['#CCCCCC','#FF6B6B','#FF9999','#4ECDC4','#88E4DF','#45B7D1','#87D4E8']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, counts, color=colors[:len(labels)], edgecolor='black', linewidth=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            str(count), ha='center', fontsize=10)
ax.set_title('FUNSD — Label Distribution (Train)', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

COLORS = {'B-HEADER':'red','I-HEADER':'red',
          'B-QUESTION':'blue','I-QUESTION':'blue',
          'B-ANSWER':'green','I-ANSWER':'green','O':'gray'}

fig, axes = plt.subplots(1, 3, figsize=(18, 10))
for idx, ax in enumerate(axes):
    s = df['train'][idx]
    img = s['image'].convert('RGB')
    d = ImageDraw.Draw(img)
    for word, box, lid in zip(s['tokens'], s['bboxes'], s['ner_tags']):
        label = ID2LABEL.get(lid, 'O')
        if label != 'O':
            d.rectangle(box, outline=COLORS[label], width=3)
    ax.imshow(img)
    ax.set_title(f'Sample {idx+1}', fontsize=12)
    ax.axis('off')

legend = [Patch(color='red',label='HEADER'),
          Patch(color='blue',label='QUESTION'),
          Patch(color='green',label='ANSWER')]
fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=12)
plt.suptitle('FUNSD — Annotated Form Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
import torch

LABEL_LIST = ['O','B-HEADER','I-HEADER','B-QUESTION','I-QUESTION','B-ANSWER','I-ANSWER']
LABEL2ID = {l:i for i,l in enumerate(LABEL_LIST)}
ID2LABEL = {i:l for i,l in enumerate(LABEL_LIST)}

MODEL_NAME = 'microsoft/layoutlmv3-base'
processor  = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)
model      = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL_LIST),
    id2label=ID2LABEL, label2id=LABEL2ID)

total = sum(p.numel() for p in model.parameters())
print(f' Model Loaded!')
print(f'   Parameters : {total:,}')
print(f'   Device     : {"GPU" if torch.cuda.is_available() else "CPU"}')

In [ ]:
import evaluate, numpy as np

seqeval = evaluate.load('seqeval')

def process_batch(batch):
    return processor(batch['image'], batch['tokens'],
                     boxes=batch['bboxes'], word_labels=batch['ner_tags'],
                     truncation=True, padding='max_length', max_length=512)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp, tl = [], []
    for pr, lr in zip(preds, labels):
        p_row, l_row = [], []
        for p, l in zip(pr, lr):
            if l != -100:
                p_row.append(ID2LABEL[p])
                l_row.append(ID2LABEL[l])
        tp.append(p_row); tl.append(l_row)
    r = seqeval.compute(predictions=tp, references=tl)
    return {'precision': round(r['overall_precision'],4),
            'recall':    round(r['overall_recall'],4),
            'f1':        round(r['overall_f1'],4),
            'accuracy':  round(r['overall_accuracy'],4)}

print('Tokenizing...')
tokenized = dataset.map(process_batch, batched=True, batch_size=4,
                        remove_columns=dataset['train'].column_names)
tokenized.set_format('torch')
print('', len(tokenized['train']), '| Test:', len(tokenized['test']))

In [ ]:
import evaluate, numpy as np

seqeval = evaluate.load('seqeval')

def process_batch(batch):
    return processor(batch['image'], batch['tokens'],
                     boxes=batch['bboxes'], word_labels=batch['ner_tags'],
                     truncation=True, padding='max_length', max_length=512)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp, tl = [], []
    for pr, lr in zip(preds, labels):
        p_row, l_row = [], []
        for p, l in zip(pr, lr):
            if l != -100:
                p_row.append(ID2LABEL[p])
                l_row.append(ID2LABEL[l])
        tp.append(p_row); tl.append(l_row)
    r = seqeval.compute(predictions=tp, references=tl)
    return {'precision': round(r['overall_precision'],4),
            'recall':    round(r['overall_recall'],4),
            'f1':        round(r['overall_f1'],4),
            'accuracy':  round(r['overall_accuracy'],4)}

print('Tokenizing...')
tokenized = df.map(process_batch, batched=True, batch_size=4,
                        remove_columns=df['train'].column_names)
tokenized.set_format('torch')
print( len(tokenized['train']), '| Test:', len(tokenized['test']))

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='layoutlmv3-finetuned',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=10,
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    tokenizer=processor,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='layoutlmv3-finetuned',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,


    eval_strategy='epoch',
    save_strategy='epoch',

    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=10,
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    tokenizer=processor,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate()

f1        = results.get('eval_f1', 0) * 100
precision = results.get('eval_precision', 0) * 100
recall    = results.get('eval_recall', 0) * 100
accuracy  = results.get('eval_accuracy', 0) * 100
